# Infant Formula Comparison: Pipeline Walkthrough

This notebook traces the full pipeline from raw PDF extraction through cleaning to visualization.
All path resolution delegates to the installed `infant_formula` package — no hardcoded paths in notebook cells.

In [16]:
import json
import pandas as pd
import plotly.express as px
import openpyxl

from infant_formula.collate import INPUTS_DIR, OUTPUTS_DIR, collate
from infant_formula.extract_pdf_tables import CONFIGS_DIR

## 1. Raw Extraction

`extract_pdf_tables.py` reads each PDF page-by-page and saves one CSV per page.
The naming convention is `{pdf_stem}_p{page}_t{table_index}.csv`.

In [3]:
csvs = sorted(OUTPUTS_DIR.glob("*_p*_t*.csv"))
print(f"{len(csvs)} page-level CSV files:\n")
for f in csvs:
    df_raw = pd.read_csv(f)
    print(f"  {f.name}: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")

5 page-level CSV files:

  Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_p1_t0.csv: 24 rows x 14 cols
  Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_p2_t0.csv: 22 rows x 14 cols
  Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_p3_t0.csv: 8 rows x 14 cols
  Consumer-Reports-Test-Results-Infant-Formula-v2_p1_t0.csv: 22 rows x 14 cols
  Consumer-Reports-Test-Results-Infant-Formula-v2_p2_t0.csv: 19 rows x 14 cols


In [ ]:
# Peek at the first page's raw extracted content
pd.read_csv(csvs[0]).head(3)

## 2. Column Configuration

The PDFs have rotated column headers that pdfplumber reads top-to-bottom (reversing each character).
`inspect_header_row()` decodes them and writes a JSON template to `configs/` for human review.
The final column names used in extraction come from these edited config files.

In [4]:
for cfg in sorted(CONFIGS_DIR.glob("*_columns.json")):
    if "original" in cfg.name:
        continue
    cols = json.loads(cfg.read_text())
    print(f"{cfg.name}:")
    for i, col in enumerate(cols):
        print(f"  {i:2d}: {col}")
    print()

Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_columns.json:
   0: brand
   1: model
   2: total_arsenic
   3: inorganic_arsenic
   4: lead
   5: cadmium
   6: mercury
   7: aluminum
   8: potassium
   9: bisphenol_a
  10: bisphenol_f
  11: bisphenol_s
  12: acrylamide

Consumer-Reports-Test-Results-Infant-Formula-v2_columns.json:
   0: brand
   1: model
   2: total_arsenic
   3: inorganic_arsenic
   4: lead
   5: cadmium
   6: mercury
   7: aluminum
   8: potassium
   9: bisphenol_a
  10: bisphenol_f
  11: bisphenol_s
  12: acrylamide



## 3. Collation & Cleaning

`collate()` does three things:
1. Concatenates all per-page CSVs into one dataframe
2. Strips thousands-separator commas from numeric strings (e.g. `"8,026,667"` → `"8026667"`)
3. Drops subheader rows (e.g. `"POWDERED"`, `"LIQUID"`) that have no `model` value

The cell below captures a raw value before `collate()` runs, then checks the cleaned version after.

In [5]:
# Skip subheader rows (e.g. "POWDERED") — they have no model value and are dropped by collate()
first_page = pd.read_csv(csvs[0])
first_data_row = first_page[first_page["model"].notna()].iloc[0]
sample_brand = first_data_row["brand"]
raw_potassium = first_data_row["potassium"]
print(f"Raw (from page CSV)  — brand: {sample_brand!r}, potassium: {raw_potassium!r}")

Raw (from page CSV)  — brand: 'Enfamil', potassium: '8,026,667'


In [6]:
collate()

collated = pd.read_csv(OUTPUTS_DIR / "all_formula.csv")
cleaned_row = collated[collated["brand"] == sample_brand].iloc[0]
print(f"After collate()      — brand: {cleaned_row['brand']!r}, potassium: {cleaned_row['potassium']!r}")
print(f"\nCollated shape: {collated.shape}")
collated.head(3)

Saved 90 rows to /Users/sarahfouzia/llm_workspace/repos/infant_formula_comparison/outputs/all_formula.csv
After collate()      — brand: 'Enfamil', potassium: '8,026,667'

Collated shape: (90, 14)


,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year
0,Enfamil,Sensitive,10.6,9.0,1.8,ND,ND,272,"8,026,667",ND,ND,ND,ND,2026
1,Enfamil,Nutramigen with Probiotic\nLGG,8.4,7.6,5.7,2.1,ND,2187,"6,970,000",8,ND,ND,12.0,2026
2,Dr. Brown’s,Good Start Soy-Ease Pro,9.9,7.9,2.3,4.3,0.5,4403,"6,806,667",ND,ND,ND,ND,2026


## 4. Input table enriched with website_link and ingredient_list

`all_formula_ingredients_manual_entry.csv`is a table with two additional columns for 'website_link' and 'ingredient_list'

Added those two columns manually as different brands render their ingredient lists differently

Using df for the output from this extraction and df_website_ingredients for the file with the links i.e. `all_formula_ingredients_manual_entry.csv`

The main reason for doing this is to keep the extraction step separate from the manual entry step. For example, if there are additional formulas tested next year, it would be better to join the existing df_website_ingredients with the additional entries from the extraction step in df. The current dataset has the chemical analyses from 2025 and 2026

In [4]:
df_website_ingredients = pd.read_csv(INPUTS_DIR / "all_formula_ingredients_manual_entry.csv")
print(f"Shape: {df_website_ingredients.shape}\n")
df_website_ingredients.dtypes

Shape: (90, 19)



index_100              int64
brand                    str
model                    str
website_link             str
ingredient_list          str
has_soy              float64
has_palm_oil         float64
total_arsenic            str
inorganic_arsenic        str
lead                     str
cadmium                  str
mercury                  str
aluminum                 str
potassium              int64
bisphenol_a              str
bisphenol_f              str
bisphenol_s              str
acrylamide               str
report_year            int64
dtype: object

In [6]:
df = pd.read_csv(OUTPUTS_DIR / "all_formula.csv")
print(f"Shape: {df.shape}\n")
df.dtypes

Shape: (90, 14)



brand                  str
model                  str
total_arsenic          str
inorganic_arsenic      str
lead                   str
cadmium                str
mercury                str
aluminum               str
potassium              str
bisphenol_a            str
bisphenol_f            str
bisphenol_s            str
acrylamide             str
report_year          int64
dtype: object

## 5. Left Join df with df_website_ingredients on brand and model

In [12]:
df = df.merge(
      df_website_ingredients[["brand", "model", "report_year", "website_link", "ingredient_list"]],
      on=["brand", "model", "report_year"],
      how="left",
  ).sort_values(["brand","model"])

In [13]:
df.head()

,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year,website_link_x,ingredient_list_x,website_link_y,ingredient_list_y
85,A2,Platinum,ND,NT,2.9,ND,ND,1770,"5,580,000",ND,ND,ND,ND,2025,https://a2platinum.com/pages/why-a2,recalled,https://a2platinum.com/pages/why-a2,recalled
89,Aptamil,First Infant Milk,ND,NT,1.5,ND,ND,215,"5,670,000",ND,ND,ND,ND,2025,https://www.nutricia.co.uk/hcp/pim-products/ap...,Dairy-based blend (of which 29% is fermented) ...,https://www.nutricia.co.uk/hcp/pim-products/ap...,Dairy-based blend (of which 29% is fermented) ...
79,Baby’s Only\nOrganic,Complete Nutrition,ND,NT,1.4,ND,ND,307,"6,080,000",ND,ND,ND,ND,2025,https://babysonly.com/collections/shop-all/pro...,"Organic Lactose, Organic Nonfat Milk (Source O...",https://babysonly.com/collections/shop-all/pro...,"Organic Lactose, Organic Nonfat Milk (Source O..."
24,Bobbie,Grass-Fed Whole Milk\n(non-organic),ND,NT,ND,ND,ND,NaN,"7,080,000",ND,ND,ND,ND,2026,NaN,NaN,NaN,NaN
81,Bobbie,Organic,ND,NT,1.2,ND,ND,1030,"6,740,000",ND,ND,ND,ND,2025,NaN,NaN,NaN,NaN


In [17]:
df_xlsx = pd.read_excel(INPUTS_DIR / "all_formula_ingredients_manual_entry.xlsx")
df_xlsx.dtypes

index_100              int64
brand                    str
model                    str
website_link             str
ingredient_list          str
has_soy              float64
has_palm_oil         float64
total_arsenic         object
inorganic_arsenic     object
lead                  object
cadmium               object
mercury               object
aluminum              object
potassium              int64
bisphenol_a           object
bisphenol_f              str
bisphenol_s              str
acrylamide            object
report_year            int64
dtype: object